In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load Silver table into Pandas
silver_df = spark.table("fraud_proj.silver_creditcard").toPandas()

# Feature engineering
silver_df["log_amount"] = np.log1p(silver_df["Amount"])
silver_df["amount_zscore"] = (silver_df["Amount"] - silver_df["Amount"].mean()) / silver_df["Amount"].std()
silver_df["time_zscore"] = (silver_df["Time"] - silver_df["Time"].mean()) / silver_df["Time"].std()

# PCA on V1–V28
v_features = [f"V{i}" for i in range(1, 29)]
scaler = StandardScaler()
v_scaled = scaler.fit_transform(silver_df[v_features])

pca = PCA(n_components=10, random_state=42)
pca_features = pca.fit_transform(v_scaled)

for i in range(pca_features.shape[1]):
    silver_df[f"PCA_{i+1}"] = pca_features[:, i]

# Convert back to Spark DF
gold_spark = spark.createDataFrame(silver_df)

# Save Gold table
# gold_spark.write.mode("overwrite").saveAsTable("fraud_proj.gold_creditcard")

gold_spark.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("fraud_proj.gold_creditcard")

display(spark.table("fraud_proj.gold_creditcard").limit(5))